#### FIFTH ATTEMPT

## Temporal Attack Prediction

### OpTC + Windows-APT datasets

**Overview:**

The combined dataset integrates OpTC endpoint telemetry with Windows-APT 2025 telemetry to provide a broader set of benign and attack-related system activities for temporal cyberattack prediction. The two datasets were harmonised into a common schema containing timestamp, event_action, event_object, protocol, process and thread identifiers (pid, ppid, tid), hostname, label, and dataset_source. 
For the LSTM and GRU experiment, timestamps are used to chronologically organise events and construct temporal sequences, allowing the models to learn patterns in preceding system activity and predict whether malicious activity will occur within a subsequent time window.

**Project goal:**

The aim is temporal prediction rather than event-level detection: use activity from the previous 10 minutes to predict whether malicious activity will occur during the next 5 minutes. The raw timestamp is retained for chronological ordering and window construction. hour and minute are also retained as model features, consistent with the previous experiment. Windows are created separately within each dataset source, hostname and date so sequences do not cross hosts, datasets or day boundaries.



In [ ]:
# Imports
%matplotlib inline

from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn import metrics

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    GRU,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

seed = 7

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [ ]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Load Combined Dataset

In [ ]:
combination_path = Path(
    "/content/drive/MyDrive/solutions/Combination_OpTC_APT"
)

train_data = pd.read_parquet(
    combination_path / "combined_train.parquet"
)

test_data = pd.read_parquet(
    combination_path / "combined_test.parquet"
)

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

Train shape: (926673, 10)
Test shape: (36550, 10)


### Prepare the timestamp and feature datatypes (Same as previous experiment)

In [ ]:
# Convert timestamps

def convert_timestamp(series):

    numeric_timestamp = pd.to_numeric(
        series,
        errors="coerce"
    )

    timestamp = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns, UTC]"
    )

    numeric_mask = numeric_timestamp.notna()

    if numeric_mask.any():

        values = numeric_timestamp[numeric_mask]

        median_value = values.abs().median()

        if median_value > 1e17:
            unit = "ns"
        elif median_value > 1e14:
            unit = "us"
        elif median_value > 1e11:
            unit = "ms"
        else:
            unit = "s"

        timestamp.loc[numeric_mask] = pd.to_datetime(
            values,
            unit=unit,
            errors="coerce",
            utc=True
        )

    string_mask = ~numeric_mask

    timestamp.loc[string_mask] = pd.to_datetime(
        series[string_mask],
        errors="coerce",
        utc=True
    )

    return timestamp


train_data["timestamp"] = convert_timestamp(
    train_data["timestamp"]
)

test_data["timestamp"] = convert_timestamp(
    test_data["timestamp"]
)

In [ ]:
# Extract time features

for data in [train_data, test_data]:

    data["hour"] = (
        data["timestamp"]
        .dt.hour
        .fillna(-1)
        .astype("float32")
    )

    data["minute"] = (
        data["timestamp"]
        .dt.minute
        .fillna(-1)
        .astype("float32")
    )

# Ensure numeric fields are numeric

for col in ["pid", "ppid", "tid"]:

    train_data[col] = pd.to_numeric(
        train_data[col],
        errors="coerce"
    ).fillna(-1).astype("float32")

    test_data[col] = pd.to_numeric(
        test_data[col],
        errors="coerce"
    ).fillna(-1).astype("float32")


# Standardise categorical fields

for col in [
    "event_action",
    "event_object",
    "protocol"
]:

    train_data[col] = (
        train_data[col]
        .fillna("UNKNOWN")
        .astype(str)
    )

    test_data[col] = (
        test_data[col]
        .fillna("UNKNOWN")
        .astype(str)
    )

### Feature Engineering

In [ ]:
# I aggregate raw events into one-minute intervals/chunks
# For each chunk, the model receives total event count, unique process/thread counts, 
# hour and minute, counts of the most common event actions, objects and protocols.

# Temporal settings
TIME_BIN = "1min"
LOOKBACK_MINUTES = 10
PREDICT_AHEAD_MINUTES = 5

TOP_ACTIONS = 20
TOP_OBJECTS = 20
TOP_PROTOCOLS = 10


# Learn categorical vocabulary from training data only
top_actions = (
    train_data["event_action"]
    .value_counts()
    .head(TOP_ACTIONS)
    .index
    .tolist()
)

top_objects = (
    train_data["event_object"]
    .value_counts()
    .head(TOP_OBJECTS)
    .index
    .tolist()
)

top_protocols = (
    train_data["protocol"]
    .value_counts()
    .head(TOP_PROTOCOLS)
    .index
    .tolist()
)


def clean_feature_name(value):
    return (
        str(value)
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(".", "_")
    )


action_columns = {
    value: f"action__{clean_feature_name(value)}"
    for value in top_actions
}

object_columns = {
    value: f"object__{clean_feature_name(value)}"
    for value in top_objects
}

protocol_columns = {
    value: f"protocol__{clean_feature_name(value)}"
    for value in top_protocols
}

In [ ]:
def build_minute_features(data):

    df = data.copy()

    df["date"] = df["timestamp"].dt.date
    df["time_bin"] = df["timestamp"].dt.floor(TIME_BIN)

    # Base numerical features
    minute_data = (
        df.groupby(
            [
                "dataset_source",
                "hostname",
                "date",
                "time_bin"
            ],
            sort=False
        )
        .agg(
            event_count=("label", "size"),
            unique_pid=("pid", "nunique"),
            unique_ppid=("ppid", "nunique"),
            unique_tid=("tid", "nunique"),
            attack_now=("label", "max")
        )
        .reset_index()
    )

    # Count selected event actions
    action_data = df[
        df["event_action"].isin(top_actions)
    ].copy()

    if len(action_data) > 0:

        action_counts = (
            action_data.groupby(
                [
                    "dataset_source",
                    "hostname",
                    "date",
                    "time_bin",
                    "event_action"
                ]
            )
            .size()
            .unstack(fill_value=0)
            .rename(columns=action_columns)
            .reset_index()
        )

        minute_data = minute_data.merge(
            action_counts,
            on=[
                "dataset_source",
                "hostname",
                "date",
                "time_bin"
            ],
            how="left"
        )

    # Count selected event objects
    object_data = df[
        df["event_object"].isin(top_objects)
    ].copy()

    if len(object_data) > 0:

        object_counts = (
            object_data.groupby(
                [
                    "dataset_source",
                    "hostname",
                    "date",
                    "time_bin",
                    "event_object"
                ]
            )
            .size()
            .unstack(fill_value=0)
            .rename(columns=object_columns)
            .reset_index()
        )

        minute_data = minute_data.merge(
            object_counts,
            on=[
                "dataset_source",
                "hostname",
                "date",
                "time_bin"
            ],
            how="left"
        )

    # Count selected protocols
    protocol_data = df[
        df["protocol"].isin(top_protocols)
    ].copy()

    if len(protocol_data) > 0:

        protocol_counts = (
            protocol_data.groupby(
                [
                    "dataset_source",
                    "hostname",
                    "date",
                    "time_bin",
                    "protocol"
                ]
            )
            .size()
            .unstack(fill_value=0)
            .rename(columns=protocol_columns)
            .reset_index()
        )

        minute_data = minute_data.merge(
            protocol_counts,
            on=[
                "dataset_source",
                "hostname",
                "date",
                "time_bin"
            ],
            how="left"
        )

    minute_data = minute_data.fillna(0)

    # Keep the same time features used previously
    minute_data["hour"] = (
        minute_data["time_bin"]
        .dt.hour
        .astype("float32")
    )

    minute_data["minute"] = (
        minute_data["time_bin"]
        .dt.minute
        .astype("float32")
    )

    return minute_data


train_minutes = build_minute_features(
    train_data
)

test_minutes = build_minute_features(
    test_data
)

print("Minute-level train shape:", train_minutes.shape)
print("Minute-level test shape:", test_minutes.shape)

### Create Target Feature

In [ ]:
def add_future_target(data):

    output = []

    group_columns = [
        "dataset_source",
        "hostname",
        "date"
    ]

    for _, group in data.groupby(
        group_columns,
        sort=False
    ):

        group = group.sort_values(
            "time_bin"
        ).copy()

        future_labels = []

        for step in range(
            1,
            PREDICT_AHEAD_MINUTES + 1
        ):

            future_labels.append(
                group["attack_now"]
                .shift(-step)
                .fillna(0)
                .to_numpy()
            )

        group["future_attack"] = (
            np.max(
                np.vstack(future_labels),
                axis=0
            ) > 0
        ).astype("int8")

        output.append(group)

    result = pd.concat(
        output,
        ignore_index=True
    )

    # Exclude minutes where the attack has already started
    result = result[
        result["attack_now"] == 0
    ].reset_index(drop=True)

    return result


train_minutes = add_future_target(
    train_minutes
)

test_minutes = add_future_target(
    test_minutes
)

print("Training target distribution:")
print(
    train_minutes["future_attack"]
    .value_counts()
)

print("\nTest target distribution:")
print(
    test_minutes["future_attack"]
    .value_counts()
)

Training shape: (926673, 8)
Test shape: (36550, 8)
Features: ['event_action', 'event_object', 'protocol', 'pid', 'ppid', 'tid', 'hour', 'minute']
Training shape: (926673, 8)
Test shape: (36550, 8)
Features: ['event_action', 'event_object', 'protocol', 'pid', 'ppid', 'tid', 'hour', 'minute']


### Scale and Encode

In [ ]:
metadata_columns = [
    "dataset_source",
    "hostname",
    "date",
    "time_bin",
    "attack_now",
    "future_attack"
]

feature_columns = [
    col
    for col in train_minutes.columns
    if col not in metadata_columns
]


# Ensure the test set has exactly the training feature space
for col in feature_columns:

    if col not in test_minutes.columns:
        test_minutes[col] = 0


X_train_minutes = train_minutes[
    feature_columns
].astype("float32")

X_test_minutes = test_minutes[
    feature_columns
].astype("float32")


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_minutes
).astype("float32")

X_test_scaled = scaler.transform(
    X_test_minutes
).astype("float32")


print(
    "Number of temporal features:",
    len(feature_columns)
)

print(
    "Train minute matrix:",
    X_train_scaled.shape
)

print(
    "Test minute matrix:",
    X_test_scaled.shape
)

(926673, 1474)
(36550, 1474)


### Create Temporal Sequences

In [ ]:
def create_sequences(
    data,
    scaled_features,
    lookback=LOOKBACK_MINUTES
):

    X_sequences = []
    y_sequences = []
    metadata = []

    group_columns = [
        "dataset_source",
        "hostname",
        "date"
    ]

    # Keep row positions aligned with the scaled matrix
    working = data.reset_index(
        drop=True
    ).copy()

    working["_row_position"] = np.arange(
        len(working)
    )

    for _, group in working.groupby(
        group_columns,
        sort=False
    ):

        group = group.sort_values(
            "time_bin"
        )

        positions = group[
            "_row_position"
        ].to_numpy()

        labels = group[
            "future_attack"
        ].to_numpy()

        times = group[
            "time_bin"
        ].to_numpy()

        sources = group[
            "dataset_source"
        ].to_numpy()

        hosts = group[
            "hostname"
        ].to_numpy()

        for end_idx in range(
            lookback - 1,
            len(group)
        ):

            start_idx = (
                end_idx - lookback + 1
            )

            window_positions = positions[
                start_idx:end_idx + 1
            ]

            X_sequences.append(
                scaled_features[
                    window_positions
                ]
            )

            y_sequences.append(
                labels[end_idx]
            )

            metadata.append({
                "dataset_source":
                    sources[end_idx],
                "hostname":
                    hosts[end_idx],
                "window_end":
                    times[end_idx]
            })

    return (
        np.asarray(
            X_sequences,
            dtype=np.float32
        ),
        np.asarray(
            y_sequences,
            dtype=np.int8
        ),
        pd.DataFrame(metadata)
    )


X_train_seq, Y_train_seq, train_seq_meta = (
    create_sequences(
        train_minutes,
        X_train_scaled
    )
)

X_test_seq, Y_test_seq, test_seq_meta = (
    create_sequences(
        test_minutes,
        X_test_scaled
    )
)


print(
    "LSTM/GRU train shape:",
    X_train_seq.shape
)

print(
    "LSTM/GRU test shape:",
    X_test_seq.shape
)

print("\nTraining sequence labels:")
print(
    pd.Series(Y_train_seq)
    .value_counts()
)

### Compute Class Weights

In [ ]:
classes = np.unique(
    Y_train_seq
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=Y_train_seq
)

class_weights = {
    int(cls): float(weight)
    for cls, weight
    in zip(classes, weights)
}

print("Class weights:", class_weights)

### Define and Train LSTM

In [ ]:
lstm_model = Sequential([
    Input(
        shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.3),

    LSTM(
        32
    ),

    BatchNormalization(),

    Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.2),

    Dense(
        1,
        activation="sigmoid"
    )
])


lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(
            name="precision"
        ),
        tf.keras.metrics.Recall(
            name="recall"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        ),
        tf.keras.metrics.AUC(
            curve="PR",
            name="pr_auc"
        )
    ]
)


callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]


lstm_history = lstm_model.fit(
    X_train_seq,
    Y_train_seq,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

### Validate on Test data

In [ ]:
lstm_prob = (
    lstm_model.predict(
        X_test_seq,
        verbose=0
    )
    .ravel()
)

lstm_pred = (
    lstm_prob >= 0.5
).astype("int8")


print(
    "LSTM Test Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        lstm_pred
    )
)

print(
    "\nLSTM Confusion Matrix:"
)

print(
    metrics.confusion_matrix(
        Y_test_seq,
        lstm_pred
    )
)

print(
    "\nLSTM Classification Report:"
)

print(
    metrics.classification_report(
        Y_test_seq,
        lstm_pred,
        digits=4,
        zero_division=0
    )
)

print(
    "LSTM ROC-AUC:",
    metrics.roc_auc_score(
        Y_test_seq,
        lstm_prob
    )
)

print(
    "LSTM PR-AUC:",
    metrics.average_precision_score(
        Y_test_seq,
        lstm_prob
    )
)


===== Logistic Regression - Test Evaluation =====
Accuracy: 0.9893296853625171
Confusion Matrix:
 [[11096   169]
 [  221 25064]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98     11265
           1       0.99      0.99      0.99     25285

    accuracy                           0.99     36550
   macro avg       0.99      0.99      0.99     36550
weighted avg       0.99      0.99      0.99     36550


===== Bernoulli NB - Test Evaluation =====
Accuracy: 0.9630916552667579
Confusion Matrix:
 [[ 9917  1348]
 [    1 25284]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.88      0.94     11265
           1       0.95      1.00      0.97     25285

    accuracy                           0.96     36550
   macro avg       0.97      0.94      0.96     36550
weighted avg       0.96      0.96      0.96     36550


===== Decision Tree - Test Evaluation =====


### Train GRU

In [ ]:
gru_model = Sequential([
    Input(
        shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),

    GRU(
        64,
        return_sequences=True
    ),

    Dropout(0.3),

    GRU(
        32
    ),

    BatchNormalization(),

    Dropout(0.3),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.2),

    Dense(
        1,
        activation="sigmoid"
    )
])


gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(
            name="precision"
        ),
        tf.keras.metrics.Recall(
            name="recall"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        ),
        tf.keras.metrics.AUC(
            curve="PR",
            name="pr_auc"
        )
    ]
)


gru_history = gru_model.fit(
    X_train_seq,
    Y_train_seq,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

DL training shape: (200000, 1474)
DL test shape: (36550, 1474)


### Validate

In [ ]:
gru_prob = (
    gru_model.predict(
        X_test_seq,
        verbose=0
    )
    .ravel()
)

gru_pred = (
    gru_prob >= 0.5
).astype("int8")


print(
    "GRU Test Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        gru_pred
    )
)

print(
    "\nGRU Confusion Matrix:"
)

print(
    metrics.confusion_matrix(
        Y_test_seq,
        gru_pred
    )
)

print(
    "\nGRU Classification Report:"
)

print(
    metrics.classification_report(
        Y_test_seq,
        gru_pred,
        digits=4,
        zero_division=0
    )
)

print(
    "GRU ROC-AUC:",
    metrics.roc_auc_score(
        Y_test_seq,
        gru_prob
    )
)

print(
    "GRU PR-AUC:",
    metrics.average_precision_score(
        Y_test_seq,
        gru_prob
    )
)

Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.9746 - loss: 0.0627 - val_accuracy: 0.9836 - val_loss: 0.0381
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9840 - loss: 0.0405 - val_accuracy: 0.9857 - val_loss: 0.0329
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9856 - loss: 0.0361 - val_accuracy: 0.9888 - val_loss: 0.0279
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9872 - loss: 0.0330 - val_accuracy: 0.9892 - val_loss: 0.0277
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9883 - loss: 0.0309 - val_accuracy: 0.9907 - val_loss: 0.0277
Epoch 6/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.9887 - loss: 0.0293 - val_accuracy: 0.9913 - val_loss: 0.0247
Epoch 7/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9894 - loss: 0.0279 - val_accuracy: 0.9916 - val_loss: 0.0240
Epoch 8/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9899 - loss: 0

### Results Presentation

In [ ]:
results = pd.DataFrame([
    {
        "Model": "LSTM",
        "Accuracy":
            metrics.accuracy_score(
                Y_test_seq,
                lstm_pred
            ),
        "Macro F1":
            metrics.f1_score(
                Y_test_seq,
                lstm_pred,
                average="macro",
                zero_division=0
            ),
        "Malicious Precision":
            metrics.precision_score(
                Y_test_seq,
                lstm_pred,
                pos_label=1,
                zero_division=0
            ),
        "Malicious Recall":
            metrics.recall_score(
                Y_test_seq,
                lstm_pred,
                pos_label=1,
                zero_division=0
            ),
        "Malicious F1":
            metrics.f1_score(
                Y_test_seq,
                lstm_pred,
                pos_label=1,
                zero_division=0
            ),
        "ROC-AUC":
            metrics.roc_auc_score(
                Y_test_seq,
                lstm_prob
            ),
        "PR-AUC":
            metrics.average_precision_score(
                Y_test_seq,
                lstm_prob
            )
    },

    {
        "Model": "GRU",
        "Accuracy":
            metrics.accuracy_score(
                Y_test_seq,
                gru_pred
            ),
        "Macro F1":
            metrics.f1_score(
                Y_test_seq,
                gru_pred,
                average="macro",
                zero_division=0
            ),
        "Malicious Precision":
            metrics.precision_score(
                Y_test_seq,
                gru_pred,
                pos_label=1,
                zero_division=0
            ),
        "Malicious Recall":
            metrics.recall_score(
                Y_test_seq,
                gru_pred,
                pos_label=1,
                zero_division=0
            ),
        "Malicious F1":
            metrics.f1_score(
                Y_test_seq,
                gru_pred,
                pos_label=1,
                zero_division=0
            ),
        "ROC-AUC":
            metrics.roc_auc_score(
                Y_test_seq,
                gru_prob
            ),
        "PR-AUC":
            metrics.average_precision_score(
                Y_test_seq,
                gru_prob
            )
    }
])

display(
    results.round(4)
)

TRAIN RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Logistic Regression,0.9140,0.7962,0.9331,0.8429
1,Bernoulli NB,0.8960,0.7690,0.8986,0.8119
2,Decision Tree,0.9724,0.9110,0.9823,0.9424
3,Random Forest,1.0000,0.9999,1.0000,1.0000
4,XGBoost,0.1259,0.0630,0.5000,0.1118
5,ANN,0.9931,0.9801,0.9889,0.9845
6,CNN,0.9921,0.9835,0.9806,0.9820


TEST RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Logistic Regression,0.9893,0.9869,0.9881,0.9875
1,Bernoulli NB,0.9631,0.9746,0.9401,0.9552
2,Decision Tree,0.9436,0.9603,0.9098,0.9303
3,Random Forest,0.9550,0.9364,0.9674,0.9492
4,XGBoost,0.6918,0.3459,0.5000,0.4089
5,ANN,0.9485,0.9284,0.9626,0.9421
6,CNN,0.9367,0.9149,0.9531,0.9293
